In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 26 — Table + Text Local RAG
## Combine CSVs and Text Documents in One RAG Pipeline

**Stack:** Ollama · LlamaIndex · pandas · Jupyter

In [ ]:
# !pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama pandas

## Step 1 — Setup

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import pandas as pd
from pathlib import Path

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")

## Step 2 — Create Mixed Data Sources

In [ ]:
Path("sample_data").mkdir(exist_ok=True)

# CSV: Sales data
sales_df = pd.DataFrame({
    "Quarter": ["Q1 2024", "Q2 2024", "Q3 2024", "Q4 2024"],
    "Revenue_M": [45.2, 52.1, 58.7, 67.3],
    "Units_Sold": [12000, 14500, 16200, 19800],
    "Top_Product": ["Widget Pro", "Widget Pro", "Widget Max", "Widget Max"],
    "Region": ["North America", "Europe", "North America", "Asia Pacific"],
})
sales_df.to_csv("sample_data/quarterly_sales.csv", index=False)

# Text: Narrative report
narrative = """
Annual Business Review 2024

Our company achieved record revenue of $223.3M in 2024, representing 35% YoY growth.
The Widget Max product line, launched in Q3, exceeded expectations with rapid adoption
in the Asia Pacific market.

Key wins included the Fortune 500 deal in Q2 ($8M ARR) and the government contract
in Q4 ($12M over 3 years). The sales team expanded from 45 to 72 people.

Challenges: Supply chain disruptions affected Q1 delivery times. Customer churn
increased to 8% from 5% due to a competitor's aggressive pricing.
"""
Path("sample_data/business_review.txt").write_text(narrative, encoding="utf-8")
print("Created CSV and text data sources")
print(sales_df.to_string(index=False))

## Step 3 — Convert Table to Text and Build Combined Index

In [ ]:
# Convert CSV rows to natural language
table_docs = []
for _, row in sales_df.iterrows():
    text = (f"In {row['Quarter']}, revenue was ${row['Revenue_M']}M "
            f"with {row['Units_Sold']:,} units sold. "
            f"Top product: {row['Top_Product']}. "
            f"Strongest region: {row['Region']}.")
    table_docs.append(Document(text=text, metadata={"source": "sales_table", "type": "data"}))

# Add summary statistics
summary_text = (f"Full year 2024: Total revenue ${sales_df['Revenue_M'].sum():.1f}M, "
               f"Total units {sales_df['Units_Sold'].sum():,}, "
               f"avg quarterly revenue ${sales_df['Revenue_M'].mean():.1f}M")
table_docs.append(Document(text=summary_text, metadata={"source": "sales_summary", "type": "data"}))

# Add narrative document
text_doc = Document(text=narrative, metadata={"source": "business_review", "type": "narrative"})

all_docs = table_docs + [text_doc]
index = VectorStoreIndex.from_documents(all_docs, show_progress=True)
query_engine = index.as_query_engine(similarity_top_k=4)
print(f"Combined index: {len(all_docs)} documents (table + text)")

## Step 4 — Query Across Tables and Text

In [ ]:
queries = [
    "What was the total revenue in 2024?",
    "Which product performed best in Q4?",
    "What were the main challenges this year?",
    "How did Asia Pacific perform?",
    "Tell me about the Fortune 500 deal.",
]

for q in queries:
    print(f"\nQ: {q}")
    response = query_engine.query(q)
    print(f"A: {response}")
    sources = [n.metadata.get('source', '?') for n in response.source_nodes]
    print(f"Sources: {sources}")

## What You Learned
- **Table-to-text conversion** for indexing structured data
- **Combined retrieval** across heterogeneous sources
- **Source attribution** showing data vs narrative evidence